In [2]:
# Get Pytorch requirement
#!pip install -q pytorch-lightning
#!pip install plotly

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import plotly.express as px # For scatter map
import requests # To create http requests for geojson
import plotly.graph_objects as go # For choropleth map

# Scikit learn packages
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression

# Time series tools
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.ar_model import AutoReg

# Torch libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pytorch_lightning as L
from sklearn.metrics import mean_absolute_error

# Creating a seed
np.random.seed(42)
torch.manual_seed(42)
L.seed_everything(42)

# Plotting parameters
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
%matplotlib inline

print('All imports successful!')

Seed set to 42


All imports successful!


In [2]:
# Dataset 
# NOTE: Change file path to local when using this notebook
csv_old = pd.read_csv("C:/Users/gabri/Documents/Chicago/Palm_watch/PalmWatch/deforestation_data/hex_deforestation_forecast_oldversion.csv")

### NOTES: Most of these functions were provided by Dr. Mario Bañuelos, in the lectures of Machine Learning Models on the second week of DSSI.

In [25]:
# Function to divide our data into train and test for modeling
def train_test_split(X, y, TRAIN_FRAC=0.8):

  """Simple test/train function

  X: Feature matrix
  y: Target Vector

  TRAIN_FRAC: If none provided, the default will be 80% split for training and 20% for test   
  This function outputs the training and testing data.
  """

  split = int(TRAIN_FRAC * len(X))
  X_train, X_test = X[:split], X[split:]
  y_train, y_test = y[:split], y[split:]

  return X_train, X_test, y_train, y_test

In [14]:
# Function to compute the RMSE and MAE for our models
def compute_metrics(y_true, y_pred, label='Model'):
    """
    Compute and print RMSE and MAE for a set of predictions.

    Parameters
    ----------
    y_true : array-like, actual values
    y_pred : array-like, predicted values
    label  : str, model name for display

    Returns
    -------
    dict with keys 'RMSE' and 'MAE'
    """
    # Compute RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # Compute MAE
    mae  = mean_absolute_error(y_true, y_pred)
    print(f'{label:<30}  RMSE={rmse:.5f}   MAE={mae:.5f}')
    return {'RMSE': rmse, 'MAE': mae}

In [16]:
# MLP class to create the model
class MLPForecaster(nn.Module):
    """
    Simple Multilayer Perceptron for time series forecasting.

    Takes a lag feature vector of length `input_size` and produces
    a single scalar forecast.

    Parameters
    ----------
    input_size  : int, number of lag features
    hidden_size : int, number of neurons in each hidden layer
    """
    def __init__(self, input_size, hidden_size=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 100),  
            nn.ReLU(), # Nonlinearly transform
            nn.Linear(100, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1)  # The output should be one neuron
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

In [17]:
# RNN (Recursive Neural Network) model
class RNNForecaster(nn.Module):
    """
    Single-layer Elman RNN for time series forecasting.

    Processes the lag sequence step-by-step and produces a single
    scalar forecast from the final hidden state.

    Parameters
    ----------
    input_size  : int, features per time step (1 for univariate)
    hidden_size : int, hidden state dimension
    """

    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()
        # Create an nn.RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)

        # Output layer mapping hidden_size → 1
        self.fc  = nn.Linear(hidden_size, 1)

    def forward(self, x):

        # x shape: (batch, n_lags) → add feature dimension
        x = x.unsqueeze(-1) # (batch, n_lags, 1)

        # Pass through RNN, extract final hidden state h_n
        _, h_n = self.rnn(x)
        h_n = h_n.squeeze(0) # (batch, hidden)

        return self.fc(h_n).squeeze(-1)

In [21]:
# LSTM model 
class LSTMForecaster(nn.Module):
    """
    Single-layer LSTM for time series forecasting.

    Extends the RNN with a cell state and gating mechanisms, allowing
    the model to selectively retain or forget information across lags.

    Parameters
    ----------
    input_size  : int, features per time step (1 for univariate)
    hidden_size : int, hidden/cell state dimension
    """

    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()
        # Create an nn.LSTM layer
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        # Output layer mapping hidden_size → 1
        self.fc   = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = x.unsqueeze(-1) # (batch, n_lags, 1)

        # Pass through LSTM, extract h_n from the returned tuple
        _, (h_n, _) = self.lstm(x)
        h_n = h_n.squeeze(0) # (batch, hidden)
        return self.fc(h_n).squeeze(-1)

In [26]:
# Function to train a time series PyTorch model
def train_model(model, X_t, y_t, epochs=300, lr=1e-3, label='Model'):
    
    """
    Train a PyTorch model with MSE loss and Adam optimizer.

    Parameters
    ----------
    model  : nn.Module
    X_t    : torch.Tensor, (n_samples, n_lags)
    y_t    : torch.Tensor, (n_samples,)
    epochs : int
    lr     : float, learning rate
    label  : str, for display

    Returns
    -------
    list of training losses
    """

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    losses    = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        # Forward pass and loss
        pred = model(X_t)
        loss = loss_fn(pred, y_t)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        losses.append(loss.item())

    print(f'{label} — final loss: {losses[-1]:.6f}')
    return losses


In [27]:
# Function to create a lag matrix for the time series
def make_lag_matrix(series, n_lags):
    
    """
    Convert a 1-D time series into a lag-embedded feature matrix.

    Parameters
    ----------
    series  : array-like, shape (T,)
    n_lags  : int, number of lag features to create

    Returns
    -------
    X : np.ndarray, shape (T - n_lags, n_lags)   <- feature matrix
    y : np.ndarray, shape (T - n_lags,)           <- target vector
    """
    
    series = np.array(series)
    T = len(series)
    X, y = [], []

    for t in range(n_lags, T):
        # Features are the previous n_lags values
        X.append(series[t-n_lags:t])
        # Target is the value at time t
        y.append(series[t])

    return np.array(X), np.array(y)

In [28]:
# Function to create a lag matrix with multiple variables of the time series
def make_lag_matrix_multi(df, n_lags, target_col):

    """
    Build a lag-embedded feature matrix from a multivariate time series.

    Parameters
    ----------
    df         : pd.DataFrame, shape (T, n_variables)
    n_lags     : int, number of lags per variable
    target_col : str, column name of the variable to forecast

    Returns
    -------
    X : pd.DataFrame of lag features
    y : pd.Series of targets
    """

    lagged_frames = []

    for lag in range(1, n_lags + 1):
        # Shift the entire DataFrame by `lag` steps
        shifted = df.shift(lag)
        shifted.columns = [f'{col}_lag{lag}' for col in df.columns]
        lagged_frames.append(shifted)

    # Combine all lagged frames side by side
    feature_df = pd.concat(lagged_frames, axis=1)

    # The target is the current (unshifted) target column
    target = df[target_col]

    # Drop rows where any lag is NaN (the first n_lags rows)
    combined = pd.concat([feature_df, target], axis=1).dropna()
    X = combined.drop(columns=[target_col])
    y = combined[target_col]

    return X, y


In [29]:
# Function to plot ACF and PACF of the time series
def ts_diagnostics(series, label='Series', lags=40, rolling_window=12):
    
    """
    Full diagnostic panel for a time series:
      1. Raw plot with rolling mean and std
      2. ADF test result shown in axis label
      3. ACF and PACF plots
    """

    series = series.dropna() # Drop NAN values in the data
    fig, axes = plt.subplots(3, 1, figsize = (12, 10))

    # Calculate rolling mean and std
    roll_mean = series.rolling(rolling_window).mean()
    roll_std = series.rolling(rolling_window).std()

    # Plot
    axes[0].plot(series, label = "Series")
    axes[0].plot(roll_mean, label = f"{rolling_window}-period Mean", color = "crimson")
    axes[0].plot(roll_std, label = f"{rolling_window}-period Std", color = "orange")
    axes[0].set_xlabel('Value')
    axes[0].set_title(f"{label}: Raw + Rolling Stats")

    adf_result = adfuller(series)
    p = adf_result[1]
    status = "STATIONARY" if p < 0.05 else "NON-STATIONARY"
    axes[0].set_xlabel(f"ADF p-value = {p:.4f} -> {status}")

    # Plot 2 ACF
    plot_acf(series, lags = lags, ax = axes[1])
    axes[1].set_title(f"{label}: ACF")

    # Plot 3 PACF
    plot_pacf(series, lags = lags, ax = axes[2], method = 'ywm')
    axes[2].set_title(f"{label}: PACF")

    plt.tight_layout()
    plt.show()

In [30]:
# RMSE calculation when comparing the test data and our forecast
def RMSE(test, preds):

  """
  Calculating the RMSE
  """

  return np.sqrt(mean_squared_error(test,preds))

In [31]:
# MAE calculation when comparing the test data and our forecast
def MAE(test, preds):

  """
  Calculating the MAE (mean absolute error)
  """

  return(mean_absolute_error(test,preds))

### Functions for Geographical Maps

In [36]:
# Function to develop a scatter map 
def scatter_map(df, color, title, size, point_title, data, map_style = "satellite", height = 1000, width = 1000):

  """
  This function creates a scatter map given a dataframe with "lat" and "lon" columns.
  If the value to group by is qualitative, but saved as an int, it is recommended to convert the column data to strings before using the map.
  .astype(str) can be used to change the data

  - df: the dataframe with the column information to plot
  - color: column name to be used when coloring or "grouping" the data points (it can be quantitative or qualitative)
  Example: Color by deforestacion fraction or cluster id
  - title: Name for the map
  - size: Name of the column with the size for each point
  - point_title: Name to appear when hovering over a point
  - data: Information to display when hovering over the points (can be a list)
  - map_style: Type of map to plot on
  - height and weight: size of the map in pixels
  """

  # Figure for the map
  fig = px.scatter_map(
    df, # Dataframe to extract information from

    # Coordinates
    lat = "lat", lon ="lon",

    hover_name = point_title, # Name for the points
    hover_data = data, # Data to display while hovering
    size = size, # Size for each point
    color = color, # Column to be used to classify and color the data

    # Colors when using discrete classification
    color_discrete_sequence = px.colors.qualitative.Light24,
    zoom = 5, # How zoomed in the map will be generated
    map_style = map_style, # Type of map
    title = title, # Map title

    # Map size
    width = width, height = height)

  fig.show()

In [37]:
# Function that loads and prepares geojson data for the choropleth map
def prep_geojson():

  """
  First function out of three to create a choropleth map of Sumatra.
  This function loads the geojson map and parses through it to obtain the coordinate information we need to plot.
  It does not need any arguments.
  Returns the original geojson, a dataframe with the geojson information of the provinces,
  and a list with the province names as found in the geojson.
  """

  # Get Indonesia geojson from github
  geojson = requests.get("https://raw.githubusercontent.com/superpikar/indonesia-geojson/master/indonesia-province-simple.json").json()

  # Parsing through the features column of the geojson
  # Features is a dictionary that contains all of the geographical information needed
  geojson_pd = pd.json_normalize(geojson["features"])

  # List to of the provinces of Sumatra as written in the geojson file
  sumatra_prov = ["DI. ACEH", "BENGKULU", "JAMBI", "LAMPUNG", "RIAU", "SUMATERA BARAT", "SUMATERA SELATAN", "SUMATERA UTARA"]

  # Only save rows with Sumatra provinces by checking if they are listed in sumatra_prov
  # properties.Propinsi contains all of the province names in Indonesia
  geojson_pdp = geojson_pd[geojson_pd["properties.Propinsi"].isin(sumatra_prov)]

  # Reset index and drop extra index column
  geojson_pdp = geojson_pdp["properties.Propinsi"].reset_index()
  geojson_pdp = geojson_pdp.drop(columns = "index")

  return geojson, geojson_pdp, sumatra_prov

In [38]:
# Function to add a column with the province geojson identification to be able to map the data
def add_prov_geoname(data_df, geo_df):

  """
  Add the official province name used in the geojson as a new column in the original dataset.
  This will be used to map the correct province boundary and region.

  NOTE: Both province names in the original dataframe and geojson should be in alphabetical order
  """

  # Get the names of the provinces in our dataframe
  provinces = data_df["province"].unique()
  counter = 0

  # Add a column with the correct province identifier to our dataset according to the geojson data
  # Iterate through the province names in our dataset
  for p in provinces:

    # Only works because the provinces are in alphabetical order
    data_df.loc[data_df["province"] == provinces[counter], "geojson"] = geo_df.iloc[counter]["properties.Propinsi"]
    counter += 1

In [40]:
# Function to create the choropleth (Use with prepared geojson and dataframe data)
def time_choropleth(df, column_to_color, time, geojson, map_title, color_scale):

  """
  Create a time animation choropleth map.

  Parameters:
  - df: Dataframe with all the yearly information by province
  - column_to_color: Column in df that will be used to color the map
  - geojson: geojson file
  - map_title: string for map name
  - color_scale: sequence of colors to use for the map
  """

  fig = px.choropleth(
    df, # Dataset with values
    geojson = geojson, # Geojson file
    locations = "geojson", # Column with geojson information in our dataset
    featureidkey = "properties.Propinsi", # Province keys in GeoJSON
    fitbounds = "locations", # Center map on Sumatra island

    # Color mapping
    color = column_to_color, # The numeric scale to shade by
    color_continuous_scale = color_scale, # Color scale to use
    animation_frame = time, # Value for the animation frame

    # Fix range throughout the years
    range_color = (min(df[column_to_color]), max(df[column_to_color])),

    title = map_title,
    width = 1500,
    height = 900
    )

  fig.show()